# Sentinel values

Some columns record a code where a measurement is missing: −999 in
an amount column, −1 in a count. The value is repeated exactly, it
sits inside a limited stretch of rows — one extract, one
instrument, one period — and it is unusual relative to the other observations.
The `sentinels` detector reports these as `missingness` findings
of type `sentinel_values`.

A column card does not separate the code from the data. The null
count reads zero, so the column looks fully observed, while the
mean, the spread, and every model fitted on the column absorb the
code as if it were a measurement.


## The test

Three conditions must hold together. The value occurs at least
20 times. Its occurrences are concentrated: the shortest window
covering at least 80% of them spans at most 25% of the column's
rows. And the value lies outside the bulk of the column's other
values, tested two ways: outside the Tukey far-out fence
(Q1 − 3 · IQR, Q3 + 3 · IQR) of those values, or across their
sign boundary — a negative value in a column whose other values
are non-negative, a positive value in a non-positive column, or
a repeated 0 in a column whose other values are strictly
positive or strictly negative.

Each condition excludes a different ordinary pattern. A genuine
value can be far out (a scattered tail value has no window), and
a genuine value can be common (a zero-inflated count's zeros sit
inside the fence). A candidate code is a value that is all three:
repeated, windowed, and outside.


In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(44)
vals = [rng.gauss(50, 10) for _ in range(2000)]
block = random.Random(7)
for i in range(600, 760):
    if block.random() < 0.5:
        vals[i] = -999.0          # one extract encoded missing as -999

df = pl.DataFrame({"amount": vals})
report = onceover.profile(df)

for f in report.findings:
    if f.get("type") == "sentinel_values":
        print(f["claim"])
        print("   evidence:", f["evidence"][0]["metrics"])


The bulk reference uses whole-column quartiles when the candidate has at
most 5% of non-null observations; above that share, quartiles are recomputed
without the candidate. At least 40 other values are required for the
exclusion-based reference. At most three candidates are reported per column.
These are heuristic missing-code candidates: confirm their meaning before
replacing observations with nulls.

## Reading the evidence

`value` is the code. `occurrences` is its count, and
`window_start_row` and `window_end_row` bound the shortest window
holding at least 80% of the occurrences. `mean_with` and
`mean_excluding` give the column mean computed with the code
treated as data and with its rows removed — the size of the
distortion the code was causing.


## What the detector does not report

- **Repeated values inside the bulk.** Zeros in a zero-inflated
  count, or a common code in a numeric column, sit inside the
  fence and are not reported: they are the column's distribution,
  not an encoding of missingness.
- **Scattered implausible values.** A far-out value spread evenly
  over the column has no covering window. The surprises pass may
  report individual extremes, but the sentinel test requires the
  window.
- **Codes below the count floor.** Fewer than 20 occurrences are
  not reported, however implausible the value: at small counts
  the window test cannot separate a code from a cluster of
  genuine values.
